# Tweet Topic Classifier — Exploration & Experiments

Compact exploration, modelling, evaluation, and deployment hand-off.

**Sections**
1. Setup
2. Data understanding (load + describe)
3. Data preparation (clean + stratified split)
4. Modelling
5. Evaluation
6. Deployment hand-off

## 1. Setup

In [2]:
# Make src importable from notebooks/.
import sys, os
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='notebook')

from src.config import load_config
from src.data.load import load_raw, label_distribution
from src.data.split import make_splits, split_summary
from src.features.preprocess import PreprocessingConfig, clean_text
from src.models.registry import DEFAULT_EXPERIMENTS, build_pipeline
from src.evaluation.metrics import score_predictions, save_confusion_matrix
from src.evaluation.compare import cross_validate_experiments, mcnemar_test

cfg = load_config()
cfg

{'random_state': 42,
 'paths': {'raw_data': 'data/raw/Data.json',
  'processed_dir': 'data/processed',
  'models_dir': 'models',
  'reports_dir': 'reports',
  'figures_dir': 'reports/figures'},
 'split': {'test_size': 0.15, 'val_size': 0.15, 'stratify': True},
 'preprocessing': {'lowercase': True,
  'strip_entity_markup': True,
  'replace_urls': True,
  'replace_mentions': True,
  'replace_hashtags': False,
  'collapse_whitespace': True},
 'cv': {'n_splits': 5, 'shuffle': True},
 'class_names': {0: 'arts_&_culture',
  1: 'business_&_entrepreneurs',
  2: 'pop_culture',
  3: 'daily_life',
  4: 'sports_&_gaming',
  5: 'science_&_technology'}}

## 2. Data understanding

Load the labelled tweets and inspect class balance, length, and dates.

In [5]:
df = load_raw()
print(f'n = {len(df):,}')
df.head()

n = 6,443


,text,date,label,id,label_name
0,The {@Clinton LumberKings@} beat the {@Cedar R...,2019-09-08,4,1170516324419866624,sports_&_gaming
1,I would rather hear Eli Gold announce this Aub...,2019-09-08,4,1170516440690176006,sports_&_gaming
2,"Someone take my phone away, I’m trying to not ...",2019-09-08,4,1170516543387709440,sports_&_gaming
3,"A year ago, Louisville struggled to beat an FC...",2019-09-08,4,1170516620466429953,sports_&_gaming
4,Anyone know why the #Dodgers #Orioles game nex...,2019-09-08,4,1170516711411310592,sports_&_gaming


In [6]:
dist = label_distribution(df)
dist

,label_name,count,pct
0,pop_culture,2512,38.99
1,sports_&_gaming,2291,35.56
2,daily_life,883,13.70
3,science_&_technology,326,5.06
4,business_&_entrepreneurs,287,4.45
5,arts_&_culture,144,2.23


In [7]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=dist.sort_values('count', ascending=False),
            x='label_name', y='count', ax=ax, color='steelblue')
ax.set_title('Class distribution')
ax.set_xlabel(''); ax.set_ylabel('count')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
fig.savefig('../reports/figures/class_distribution.png', dpi=160)
plt.show()

/var/folders/tl/0y2j20r51dg5lmj0swk6lh8h0000gn/T/ipykernel_52161/1108272148.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
df['n_chars'] = df['text'].str.len()
df['n_tokens'] = df['text'].str.split().str.len()
df[['n_chars', 'n_tokens']].describe()

,n_chars,n_tokens
count,6443.000000,6443.000000
mean,166.002949,27.681360
std,68.213074,12.303891
min,38.000000,6.000000
25%,107.000000,17.000000
50%,155.000000,25.000000
75%,224.000000,37.000000
max,353.000000,62.000000


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df['n_chars'], bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('Characters per tweet')
sns.histplot(df['n_tokens'], bins=30, ax=axes[1], color='steelblue')
axes[1].set_title('Whitespace-split tokens per tweet')
plt.tight_layout()
fig.savefig('../reports/figures/length_distribution.png', dpi=160)
plt.show()

/var/folders/tl/0y2j20r51dg5lmj0swk6lh8h0000gn/T/ipykernel_52161/1130683435.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Data preparation

Sanity-check the light text cleaning against raw examples.

In [10]:
sample = df['text'].sample(3, random_state=0).tolist()
for s in sample:
    print('RAW :', s)
    print('CLEAN:', clean_text(s, PreprocessingConfig.from_dict(cfg['preprocessing'])))
    print('---')

RAW : happy anniversary my free therapy Run {@BTS_twt@}
CLEAN: happy anniversary my free therapy run bts_twt
---
RAW : happy new year   everyone out there my best memory of 2020 is that I got to talk to two wonderful people   and one of them I love from the bottom of my heart {{USERNAME}} {{USERNAME}}
CLEAN: happy new year everyone out there my best memory of 2020 is that i got to talk to two wonderful people and one of them i love from the bottom of my heart {{username}} {{username}}
---
RAW : I read the Strike novels by {@J K Rowling@} when they came out. her feelings about trans women are obvious in 2 & 3. 2 has a trans character who stabs people because of their trans trauma. book 3 has  jokes  about not passing and compares being trans with  body integrity disorder
CLEAN: i read the strike novels by j k rowling when they came out. her feelings about trans women are obvious in 2 & 3. 2 has a trans character who stabs people because of their trans trauma. book 3 has jokes about not 

In [11]:
splits = make_splits()
split_summary(splits)

,split,n,pop_culture,sports_&_gaming,daily_life,science_&_technology,business_&_entrepreneurs,arts_&_culture
0,train,4509,1758,1603,618,228,201,101
1,val,967,377,344,133,49,43,21
2,test,967,377,344,132,49,43,22


## 4. Modelling — representations × classifiers

Fit the registry experiments on `train`, score on `val`, and rank by macro-F1.

In [ ]:
train, val, test = splits['train'], splits['val'], splits['test']
class_names = [cfg['class_names'][i] for i in sorted(cfg['class_names'])]

fitted = {}
rows = []
for spec in DEFAULT_EXPERIMENTS:
    pipe = build_pipeline(spec)
    pipe.fit(train['text'].tolist(), train['label'].to_numpy())
    y_pred = pipe.predict(val['text'].tolist())
    m = score_predictions(val['label'].to_numpy(), y_pred, class_names)
    rows.append({
        'experiment': spec.name,
        'representation': spec.representation,
        'classifier': spec.classifier,
        'accuracy': m['accuracy'],
        'macro_f1': m['macro_f1'],
        'weighted_f1': m['weighted_f1'],
    })
    fitted[spec.name] = pipe
    print(f"{spec.name:30s}  acc={m['accuracy']:.4f}  macro_f1={m['macro_f1']:.4f}")

val_leaderboard = pd.DataFrame(rows).sort_values('macro_f1', ascending=False).reset_index(drop=True)
val_leaderboard

### 4b. 5-fold stratified cross-validation

Run stratified 5-fold CV on the training set for a more stable ranking.

In [ ]:
cv_df = cross_validate_experiments()
cv_df

In [ ]:
# Validation macro-F1 vs CV mean macro-F1.
merged = val_leaderboard.merge(
    cv_df[['experiment', 'f1_macro_mean', 'f1_macro_std']],
    on='experiment'
).sort_values('f1_macro_mean', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
y = np.arange(len(merged))
ax.errorbar(merged['f1_macro_mean'], y, xerr=merged['f1_macro_std'],
            fmt='o', color='steelblue', ecolor='gray', capsize=4, label='5-fold CV')
ax.scatter(merged['macro_f1'], y, marker='x', color='crimson', label='single val split')
ax.set_yticks(y); ax.set_yticklabels(merged['experiment'])
ax.set_xlabel('macro F1')
ax.set_title('Experiment comparison: 5-fold CV vs single split')
ax.legend()
plt.tight_layout()
fig.savefig('../reports/figures/experiment_comparison.png', dpi=160)
plt.show()

## 5. Evaluation on the held-out test set

Pick the CV winner, refit on `train + val`, test once, then compare with McNemar.

In [ ]:
winner_name = cv_df.iloc[0]['experiment']
runner_up_name = cv_df.iloc[1]['experiment']
print('Winner   :', winner_name)
print('Runner-up:', runner_up_name)

In [ ]:
combined = pd.concat([train, val], ignore_index=True)

def refit_and_predict(name):
    spec = next(s for s in DEFAULT_EXPERIMENTS if s.name == name)
    pipe = build_pipeline(spec)
    pipe.fit(combined['text'].tolist(), combined['label'].to_numpy())
    return pipe, pipe.predict(test['text'].tolist())

winner_pipe,    winner_pred    = refit_and_predict(winner_name)
runner_up_pipe, runner_up_pred = refit_and_predict(runner_up_name)

winner_metrics = score_predictions(test['label'].to_numpy(), winner_pred, class_names)
runner_metrics = score_predictions(test['label'].to_numpy(), runner_up_pred, class_names)
print(f"{winner_name}    test: acc={winner_metrics['accuracy']:.4f}  macro_f1={winner_metrics['macro_f1']:.4f}")
print(f"{runner_up_name} test: acc={runner_metrics['accuracy']:.4f}  macro_f1={runner_metrics['macro_f1']:.4f}")

In [ ]:
# Per-class report table.
per_class = pd.DataFrame(winner_metrics['per_class']).T
per_class = per_class[['precision', 'recall', 'f1', 'support', 'tp', 'fp', 'fn', 'tn']]
per_class.round(3)

In [ ]:
# Winner confusion matrix on test.
save_confusion_matrix(
    y_true=test['label'].to_numpy(),
    y_pred=winner_pred,
    class_names=class_names,
    title=f'{winner_name} — held-out test',
    out_path=f'../reports/figures/cm_test_{winner_name}.png',
)
from IPython.display import Image
Image(f'../reports/figures/cm_test_{winner_name}.png')

In [ ]:
mc = mcnemar_test(test['label'].to_numpy(), winner_pred, runner_up_pred)
mc

## 6. Deployment hand-off

Persist the fitted winner for the Streamlit app. The training script reproduces this flow.

In [ ]:
import joblib, json
from datetime import datetime, timezone
from src.config import resolve_path

models_dir = resolve_path(cfg['paths']['models_dir']); models_dir.mkdir(exist_ok=True, parents=True)
joblib.dump(winner_pipe, models_dir / 'best_pipeline.joblib')
winner_spec = next(s for s in DEFAULT_EXPERIMENTS if s.name == winner_name)
(models_dir / 'best_pipeline.meta.json').write_text(json.dumps({
    'trained_at_utc': datetime.now(timezone.utc).isoformat(),
    'experiment': winner_name,
    'representation': winner_spec.representation,
    'classifier': winner_spec.classifier,
    'class_names': class_names,
    'test_metrics': {k: winner_metrics[k] for k in ('accuracy', 'macro_f1', 'weighted_f1')},
}, indent=2))
print('Persisted.')